# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Agha314/FLyRank-Task-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_Token')}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# Pull the same per-page March aggregates you already trust from w03
page_month = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_gsc_impressions_mar,
        SUM(gsc_clicks)      AS total_gsc_clicks_mar,
        AVG(gsc_avg_position) AS avg_gsc_position_mar
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id
""").df()

# Position tiers — same shape as the product's own tiering logic
import pandas as pd
bins   = [0, 3, 10, 20, float("inf")]
labels = ["1-3", "4-10", "11-20", "21+"]
page_month["position_tier"] = pd.cut(page_month["avg_gsc_position_mar"], bins=bins, labels=labels)

# Weighted CTR per tier — SUM(clicks)/SUM(impressions), NOT the mean of per-page ctr_mar
bucket_table = page_month.groupby("position_tier",observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "weighted_ctr_pct": 100 * g["total_gsc_clicks_mar"].sum() / g["total_gsc_impressions_mar"].sum()
    })
).reset_index()

print(bucket_table)
# Signal 2 — Volume (impressions) vs decline rate, tested at FlyRank's real quick-win floor (>=100 impressions)
page_month_full = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_gsc_impressions_mar,
        SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15') AS first_half_clicks,
        SUM(gsc_clicks) FILTER (WHERE report_date > DATE '2026-03-15')  AS second_half_clicks,
        CASE WHEN SUM(gsc_clicks) FILTER (WHERE report_date > DATE '2026-03-15')
                  < SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15')
             THEN 1 ELSE 0 END AS is_declining
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id
""").df()

page_month_full["volume_tier"] = page_month_full["total_gsc_impressions_mar"].apply(
    lambda x: "below 100 (low demand)" if x < 100 else "100+ (real demand)"
)

volume_bucket_table = page_month_full.groupby("volume_tier", observed=True).agg(
    n=("is_declining", "size"),
    decline_rate_pct=("is_declining", lambda x: 100 * x.mean())
).reset_index()

print(volume_bucket_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_1224/2124868033.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bucket_table = page_month.groupby("position_tier",observed=True).apply(


  position_tier        n  weighted_ctr_pct
0           1-3  16144.0          0.406253
1          4-10  81988.0          0.323895
2         11-20  32203.0          0.305241
3           21+  44969.0          0.133142


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

              volume_tier       n  decline_rate_pct
0      100+ (real demand)  101441         26.419298
1  below 100 (low demand)  229996          0.951756


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.